In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('mlflw').getOrCreate()

26/09/20 20:04:43 WARN Utils: Your hostname, Sidharthas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.100 instead (on interface en0)
26/09/20 20:04:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 20:04:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/20 20:04:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
data = [
    (32, 78, 12.5, 1012, 0),
    (35, 72, 10.2, 1010, 0),
    (38, 65, 8.5, 1008, 0),
    (40, 60, 7.2, 1006, 0),
    (42, 55, 6.8, 1005, 0),

    (28, 88, 18.2, 1009, 1),
    (30, 91, 20.5, 1007, 1),
    (26, 94, 22.1, 1004, 1),
    (24, 89, 19.8, 1003, 1),
    (22, 92, 21.5, 1001, 1),

    (45, 48, 5.2, 1015, 0),
    (47, 45, 4.8, 1017, 0),
    (50, 42, 4.2, 1018, 0),
    (52, 40, 3.8, 1020, 0),
    (55, 38, 3.2, 1022, 0),

    (27, 85, 17.5, 1005, 1),
    (29, 87, 19.2, 1002, 1),
    (31, 90, 21.0, 1000, 1),
    (25, 93, 23.5, 998, 1),
    (23, 95, 24.2, 997, 1),

    (36, 70, 11.5, 1011, 0),
    (39, 67, 9.8, 1009, 0),
    (41, 63, 8.2, 1007, 0),
    (43, 58, 7.5, 1006, 0),
    (46, 52, 6.0, 1014, 0),

    (26, 86, 18.8, 1004, 1),
    (28, 89, 20.2, 1001, 1),
    (30, 92, 22.5, 999, 1),
    (21, 96, 25.1, 996, 1),
    (20, 94, 24.8, 995, 1)
]

columns = [
    "temperature",
    "humidity",
    "wind_speed",
    "pressure",
    "label"
]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

+-----------+--------+----------+--------+-----+
|temperature|humidity|wind_speed|pressure|label|
+-----------+--------+----------+--------+-----+
|         32|      78|      12.5|    1012|    0|
|         35|      72|      10.2|    1010|    0|
|         38|      65|       8.5|    1008|    0|
|         40|      60|       7.2|    1006|    0|
|         42|      55|       6.8|    1005|    0|
|         28|      88|      18.2|    1009|    1|
|         30|      91|      20.5|    1007|    1|
|         26|      94|      22.1|    1004|    1|
|         24|      89|      19.8|    1003|    1|
|         22|      92|      21.5|    1001|    1|
|         45|      48|       5.2|    1015|    0|
|         47|      45|       4.8|    1017|    0|
|         50|      42|       4.2|    1018|    0|
|         52|      40|       3.8|    1020|    0|
|         55|      38|       3.2|    1022|    0|
|         27|      85|      17.5|    1005|    1|
|         29|      87|      19.2|    1002|    1|
|         31|      9

In [8]:
train, test = df.randomSplit([0.8, 0.2], seed=42)

In [9]:
print("Training rows:", train.count())
print("Testing rows:", test.count())

Training rows: 25
Testing rows: 5


In [10]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "temperature",
        "humidity",
        "wind_speed",
        "pressure"
    ],
    outputCol="features"
)

In [11]:
train_features = assembler.transform(train)

train_features.show()

+-----------+--------+----------+--------+-----+--------------------+
|temperature|humidity|wind_speed|pressure|label|            features|
+-----------+--------+----------+--------+-----+--------------------+
|         32|      78|      12.5|    1012|    0|[32.0,78.0,12.5,1...|
|         35|      72|      10.2|    1010|    0|[35.0,72.0,10.2,1...|
|         40|      60|       7.2|    1006|    0|[40.0,60.0,7.2,10...|
|         42|      55|       6.8|    1005|    0|[42.0,55.0,6.8,10...|
|         24|      89|      19.8|    1003|    1|[24.0,89.0,19.8,1...|
|         30|      91|      20.5|    1007|    1|[30.0,91.0,20.5,1...|
|         22|      92|      21.5|    1001|    1|[22.0,92.0,21.5,1...|
|         45|      48|       5.2|    1015|    0|[45.0,48.0,5.2,10...|
|         47|      45|       4.8|    1017|    0|[47.0,45.0,4.8,10...|
|         50|      42|       4.2|    1018|    0|[50.0,42.0,4.2,10...|
|         52|      40|       3.8|    1020|    0|[52.0,40.0,3.8,10...|
|         55|      3

In [12]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    seed=42
)

In [14]:
model = rf.fit(train_features)

26/09/20 20:42:32 WARN DecisionTreeMetadata: DecisionTree reducing maxBins from 32 to 25 (= number of training instances)


In [15]:
print(model)

RandomForestClassificationModel: uid=RandomForestClassifier_ff2031dba003, numTrees=100, numClasses=2, numFeatures=4


In [16]:
test_features = assembler.transform(test)

In [17]:
test_features.show()

+-----------+--------+----------+--------+-----+--------------------+
|temperature|humidity|wind_speed|pressure|label|            features|
+-----------+--------+----------+--------+-----+--------------------+
|         38|      65|       8.5|    1008|    0|[38.0,65.0,8.5,10...|
|         28|      88|      18.2|    1009|    1|[28.0,88.0,18.2,1...|
|         26|      94|      22.1|    1004|    1|[26.0,94.0,22.1,1...|
|         31|      90|      21.0|    1000|    1|[31.0,90.0,21.0,1...|
|         43|      58|       7.5|    1006|    0|[43.0,58.0,7.5,10...|
+-----------+--------+----------+--------+-----+--------------------+



In [18]:
predictions = model.transform(test_features)

In [19]:
predictions.select(
    "temperature",
    "humidity",
    "wind_speed",
    "pressure",
    "label",
    "prediction",
    "probability"
).show(truncate=False)

+-----------+--------+----------+--------+-----+----------+-----------+
|temperature|humidity|wind_speed|pressure|label|prediction|probability|
+-----------+--------+----------+--------+-----+----------+-----------+
|38         |65      |8.5       |1008    |0    |0.0       |[0.99,0.01]|
|28         |88      |18.2      |1009    |1    |1.0       |[0.07,0.93]|
|26         |94      |22.1      |1004    |1    |1.0       |[0.02,0.98]|
|31         |90      |21.0      |1000    |1    |1.0       |[0.06,0.94]|
|43         |58      |7.5       |1006    |0    |0.0       |[1.0,0.0]  |
+-----------+--------+----------+--------+-----+----------+-----------+



In [20]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [21]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

In [22]:
accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 1.0


In [23]:
model.write().overwrite().save("model_dir/weather_rf")

In [24]:
from pyspark.ml.classification import RandomForestClassificationModel

In [25]:
loaded_model = RandomForestClassificationModel.load(
    "model_dir/weather_rf"
)
loaded_predictions = loaded_model.transform(test_features)
loaded_predictions.select(
    "label",
    "prediction",
    "probability"
).show(truncate=False)

+-----+----------+-----------+
|label|prediction|probability|
+-----+----------+-----------+
|0    |0.0       |[0.99,0.01]|
|1    |1.0       |[0.07,0.93]|
|1    |1.0       |[0.02,0.98]|
|1    |1.0       |[0.06,0.94]|
|0    |0.0       |[1.0,0.0]  |
+-----+----------+-----------+



In [ ]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run():

    mlflow.spark.log_model(
        model,
        "weather_rf"
    )

model_uri = f"runs:/{mlflow.active_run().info.run_id}/weather_rf"
mlflow.register_model(
    model_uri=model_uri,
    name="my_catalog.my_schema.weather_rf"
)

mlflow.register_model(
    model_uri=model_uri,
    name="my_catalog.my_schema.weather_rf"
)

model = mlflow.spark.load_model(
    "models:/my_catalog.my_schema.weather_rf/1"
)

predictions = model.transform(test_features)